In [1]:
pip install torch torchvision numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 46.9 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 35.7 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 72.1.0
    Uninstalling setuptools-72.1.0:
      Successfully uninstalled setuptools-72.1.0 0/3 [setuptools]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchvision] [torchvision]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

device = (
    torch.device("mps")
    if torch.backends.mps.is_available()
    else torch.device("cuda")
    if torch.cuda.is_available()
    else torch.device("cpu")
)

print(f"Using device: {device}")

Using device: mps


In [9]:
class AssignmentCNN(nn.Module):
    def __init__(self):
        super(AssignmentCNN, self).__init__()

        # First convolutional layer
        self.conv1 = nn.Conv2d(
            3, 16, kernel_size=3, stride=1, padding=1
        )

        # Max pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Second convolutional layer
        self.conv2 = nn.Conv2d(
            16, 32, kernel_size=3, stride=1, padding=1
        )

        # Fully connected layers
        self.fc1 = nn.Linear(32 * 16 * 16, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


In [10]:
# Create the CNN model
model = AssignmentCNN().to(device)

# Create a sample RGB image (64x64)
dummy_input = torch.randn(1, 3, 64, 64).to(device)

# Forward pass
output = model(dummy_input)

# Check input and output shapes
print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)

# Display model architecture
print(model)


Input shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 10])
AssignmentCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=8192, out_features=100, bias=True)
  (fc2): Linear(in_features=100, out_features=10, bias=True)
)


In [11]:

# Verify output shapes for each layer
x = dummy_input

x = F.relu(model.conv1(x))
print("After Conv1:", x.shape)

x = model.pool(x)
print("After Pool1:", x.shape)

x = F.relu(model.conv2(x))
print("After Conv2:", x.shape)

x = model.pool(x)
print("After Pool2:", x.shape)

x = torch.flatten(x, 1)
print("After Flatten:", x.shape)

x = F.relu(model.fc1(x))
print("After FC1:", x.shape)

x = model.fc2(x)
print("After FC2:", x.shape)


After Conv1: torch.Size([1, 16, 64, 64])
After Pool1: torch.Size([1, 16, 32, 32])
After Conv2: torch.Size([1, 32, 32, 32])
After Pool2: torch.Size([1, 32, 16, 16])
After Flatten: torch.Size([1, 8192])
After FC1: torch.Size([1, 100])
After FC2: torch.Size([1, 10])


In [12]:

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Resize CIFAR10 images to 64x64
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Load training dataset
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Load testing dataset
test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))

images, labels = next(iter(train_loader))
print("Batch shape:", images.shape)


100%|██████████| 170M/170M [38:31<00:00, 73.7kB/s] 


Training images: 50000
Testing images: 10000
Batch shape: torch.Size([32, 3, 64, 64])


In [13]:
# CIFAR10 class labels
CLASSES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

print("CIFAR10 Classes:", CLASSES)

CIFAR10 Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [15]:
import torch.optim as optim

# Initialize the CNN model
model = AssignmentCNN().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

# Test with 1 epoch
EPOCHS = 10

train_losses = []
train_accuracies = []

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch+1}/{EPOCHS}], "
        f"Loss: {epoch_loss:.4f}, "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

print("Finished Training!")


Epoch [1/10], Loss: 1.5407, Accuracy: 44.93%
Epoch [2/10], Loss: 1.2371, Accuracy: 55.95%
Epoch [3/10], Loss: 1.1185, Accuracy: 60.50%
Epoch [4/10], Loss: 1.0289, Accuracy: 63.74%
Epoch [5/10], Loss: 0.9575, Accuracy: 66.22%
Epoch [6/10], Loss: 0.8967, Accuracy: 68.45%
Epoch [7/10], Loss: 0.8382, Accuracy: 70.41%
Epoch [8/10], Loss: 0.7776, Accuracy: 72.56%
Epoch [9/10], Loss: 0.7231, Accuracy: 74.54%
Epoch [10/10], Loss: 0.6657, Accuracy: 76.43%
Finished Training!


In [16]:
# Evaluate CNN on CIFAR10 test dataset
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total

print(f"Test Accuracy: {test_accuracy:.2f}%")


Test Accuracy: 63.91%


In [17]:
torch.save(model.state_dict(), "assignment2_cnn.pth")

print("Model saved successfully!")


Model saved successfully!


In [18]:

import os

print("Current directory:", os.getcwd())

print(
    "Model file exists:",
    os.path.exists("assignment2_cnn.pth")
)

print(
    "Model file size (MB):",
    round(os.path.getsize("assignment2_cnn.pth") / 1024**2, 2)
)


Current directory: /Users/liuyue/capstone
Model file exists: True
Model file size (MB): 3.15
